# పాఠం 18: క్రీప్టోగ్రాఫిక్ రసీదులతో AI ఏజెంట్లను సెక్యూరు చేయడం

## హ్యాండ్స్-ఆన్ నోట్‌బుక్

ఈ నోట్‌బుక్ నాలుగు వ్యాయామాలను చూపిస్తుంది:

1. ఏజెంట్ టూల్ కాల్ కోసం మీ మొదటి రసీదును సైన్ చేయండి మరియు దాన్ని ధృవీకరించండి.
2. రసీదుతో మోసపూరిత మార్పులు చేయండి మరియు ధృవీకరణ విఫలమవడం చూడండి.
3. మూడు రసీదు చైనుని నిర్మించి చైన్ సమగ్రతను నిర్ధారించండి.
4. మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్ టూల్ కాల్‌ను ర్యాప్ చేయండి, ప్రతి చర్య రసీదు విడుదల చేస్తుంది.

అన్ని క్రీప్టోగ్రాఫిక్ ప్రిమిటివ్స్‌ మరమ్మతులైన లైబ్రరీల నుండి దిగుమతి చేసుకున్నారు (`pynacl` Ed25519 కోసం, `jcs` RFC 8785 కనానికల్ JSON కోసం, `hashlib` SHA-256 కోసం, Python స్టాండర్డ్ లైబ్రరీ నుంచి). రసీదు లాజిక్ స్వయంగా సాదా Python‌లో ఉండి మీరు చదవవచ్చు, మార్చవచ్చు.

క్రమం తప్పకుండా సెల్స్ నడిపించండి. ప్రతి భాగం చిన్నది మరియు స్వయం-సంసిద్ధంగా ఉంటుంది.


## సెటప్

రెండు డిపెండెన్సీలు ఇన్‌స్టాల్ చేయండి. రెండింటికి అనుమతిపరమైన లైసెన్సులు ఉన్నాయి (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## సహాయక సహాయ పరికరాలు

ఈ రెండు సహాయక పరికరాలు బేస్64url ఎన్‌కోడింగ్ (ప్యాడింగ్ లేకుండా) మరియు ఏదైనా వస్తువుల SHA-256 హాషింగ్‌ను నిర్వహిస్తాయి. అవి ఇంకొన్ని అంశాలు కాకుండా రసీదు లాజిక్ పైనే నోట్‌బుక్‌ను కేంద్రీకృతం చేస్తాయి.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## సెక్షన్ 1: మీ మొదటి రిసిప్ట్‌ను సైన్ చేయండి

మన **Contoso ట్రావెల్** ఏజెంట్ సిడ్నీ నుండి లాస్ ఏంజిల్స్‌కు ప్రయాణిస్తోన్న ఒక కస్టమర్ కోసం ఫ్లైట్స్‌ను మాత్రమే చూడటాన్ని ఊహించుకోండి. భవిష్యత్తులో ఒక ఆడిటర్ మాపై నమ్మకం పెట్టుకోకుండానే ధ్రువీకరించగలిగేలా, ఈ టూల్ కాల్‌ను సంతకం చేయబడిన రిసిప్ట్‌గా నమోదు చేయాలనుకుంటున్నాము.

### దశ 1.1: సైన్ చేసే కీని రూపొందించండి

ప్రొడక్షన్‌లో, ఏజెంట్ యొక్క సైన్ చేసే కీ ఒక హార్డ్‌వేర్ సెక్యూరిటీ మాడ్యూల్ (HSM), Azure Key Vault లేదా అనురూప రక్షిత స్టోర్లో ఉండేది. ఈ పాఠంలో మేము మెమరీలో తాజా కీని రూపొందించుకుంటాము.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### దశ 1.2: రసీదు పేఇలోడ్‌ను నిర్మించండి

పేఇలోడ్‌లో రసీదు సాక్ష్యం వహించాల్సిన ప్రతిదీ ఉంటుంది: ఎవరు చర్య తీసుకున్నారు, ఏ పరికరాన్ని ఉపయోగించారు, ఏ వాదనలు ఉన్నాయో, ఏం తిరిగి వచ్చింది, ఏ విధానంతో మరియు ఎప్పుడు. రసీదు సున్నితమైన విషయాలను బయటపడకుండా ఉండేందుకు వాదనలతో ఫలితాలను ఇంట్లోపల ఉంచకుండానే వాటిని హాష్ చేస్తాము.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### దశ 1.3: రసీదుపై సంతకం చేయండి మరియు కూడగట్టండి

మూడుసార్లు:

1. అదే తార్కిక రసీదును ఉత్పత్తి చేసే రెండు అమలు byte-అచ్చుగా బిట్లను ఉత్పత్తి చేయడానికి JCS ఉపయోగించి లోడ్ను శ్రేణీకరణ చేయండి.
2. SHA-256తో శ్రేణీకృత బిట్లను హ్యాష్ చేయండి.
3. Ed25519 ప్రైవేట్ కీతో హ్యాష్‌పై సంతకం చేయండి.

ఆపై సంతకం మొదటి లోడ్కు జతచేయబడుతుంది తద్వారా తుది రసీదు రూపొందుతుంది.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### దశ 1.4: రసీదు ధృవీకరణ

ధృవీకరణ ప్రక్రియను తిరగబడుతుంది. మేము సంతకాన్ని తీసివేస్తాము, క్యానానికల్ హ్యాష్‌ను పునః గణన చేస్తాము, మరియు రసీదులో ఉన్న తాజాగా సార్తకమైన సంతకం తో సరిపోల్చుతాము.

ఈ ధృవీకరణను చేయడం కోసం ఆడిటర్‌కు కేవలం రసీదు తప్ప మాకు నుండి ఇంకేమీ అవసరం ఉండదు. సేవను కాల్ చేయడం, కీ డైరెక్టరీ ను ప్రశ్నించడం లేదా విశ్వాసం అవసరం లేదు.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

మీరు `Receipt is valid: True` కనిపించాలి. ఏజెంట్ తన మొదటి క్రిప్టోగ్రాఫిక్ సంతకం చేసిన ఆడిట్ రికార్డ్ ని ఉత్పత్తి చేసింది.


## విభాగం 2: రసీదు తో తొత్తరించడం

రసీదుల యొక్క మొత్తం ఉద్దేశ్యం అవి తొత్తరించకపోవడం. దీనిని ప్రూవ్ చేయుదాం.

మనం రసీదులో ఒకే ఒక్క అక్షరాన్ని మార్చి ధృవీకరణ విఫలమవడం చూడవచ్చు.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### ఈ మధ్య ఏమయ్యింది?

మేము `policy_id`ని మార్చినప్పుడు, కానానికల్ బైట్స్ మారాయి. ఆ బైట్స్ యొక్క SHA-256 హ్యాష్ మారింది. ఆ సంతకం (మూల హ్యాష్ మీద చేసింది) ఇక కొత్త హ్యాష్‌తో సరిపోదు. నిర్ధారణ సరైనంగా `False` ను తిరిగి ఇస్తుంది.

రసీదులో ఏవైనా ఫీల్డ్‌లను మార్చి ఇంకా అది సరైనదేనని నిర్ధారించలేము, ఆక్రమణకర్త ప్రైవేట్ కీ లేకపోతే. ప్రైవేట్ కీ కీ వాల్ట్‌లో ఉంటే మరియు పబ్లిక్ కీ ప్రచురించబడితే, తప్పుడు చేయడం దాచలేరు.

మీరు ప్రయత్నించండి: పై సెల్‌లో `tool_name` లేదా `agent_id` లేదా `timestamp` మార్చి తిరిగి నడిపించండి. ప్రతి మార్పు చెత్త రసీదును ఇస్తుంది.


## విభాగం 3: రశీదులను సరి పెట్టడం

ఒకే రశీదు ఒక చర్యను రక్షిస్తుంది. చాలా ఏజెంట్లు అనేక చర్యలను తీసుకుంటారు. మొత్తం సీక్వెన్స్ టాంపర్-స్పష్టంగా ఉండేలా చేయడానికి, మేము ప్రతి రశీదును మునుపటి రశీదు యొక్క హాష్‌ను కొత్త రశీదు లోడ్‌లో చేర్చడం ద్వారా ముందటి రశీదుకు జత చేస్తాము.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

ఎవ్వరైనా రశీదును తీసివేస్తే లేదా తిరిగి వ్యవస్థ చేయించినా, చైన్ అక్షరం అంతే చోట విరుగుతుంది. తర్వాతి ఏదైనా రశీదుకు ప్రమాణీకరణ విఫలమవుతుంది ఎందుకంటే అది `previous_receipt_hash` అసలు సరిపోలకపోయినదాన్ని తన పూర్వదానికి సంబంధించిన హాష్ కి సరిపోలదు.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

ఇప్పుడు మధ్య లైబ్రరీని తగలబడటంతో గొలుసును రద్దు చేసి మళ్లీ పరీక్షించండి. తగలబడిన రసీదును దాని సంతకం తనిఖీ విఫలమవుతుంది, మరియు తరువాతి రసీదు గొలుసు-లింక్ తనిఖీ విఫలమవుతుంది (ఎందుకంటే దాని `previous_receipt_hash` ఇకపై మార్చబడిన మధ్య రసీదు యొక్క హాష్‌కు సరిపోడం లేదు).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

రసీదు 0 ఇంకా ధృవీకరిస్తోంది (అది మార్చబడలేదు మరియు ఆధారపడే ముందరి వర్గం లేదు). రసీదు 1 తన సంతకం తనిఖీ విఫలమవుతుంది ఎందుకంటే మేము `tool_args_hash` మార్చాము. రసీదు 2 తన శృంఖల-లింక్ తనిఖీ విఫలమవుతుంది ఎందుకంటే దాని `previous_receipt_hash` అసలు (ఇప్పుడు-మార్చబడిన) రసీదు 1 మీద ఆధారపడి గణించబడింది.

ఒక దాడి చేసినవారు మార్చిన రసీదు 1 ను మళ్ళీ సంతకం చేసినా (అది ప్రైవేటు కీ లేకుండా చేయలేరు అయినప్పటికీ), రసీదు 2లో శృంఖల-లింక్ విభిన్నత ఇంకా జాలరి గోప్యతను వెల్లడిస్తుంది. మార్పును దాచడానికి, దాడి చేసే వారు మార్పు పాయింట్ నుండి ప్రతి రసీదు మళ్లీ సంతకం చేయవలసి ఉంటుంది, ఇది ప్రైవేటు కీ కలిగి ఉండటం అవసరం.


## విభాగం 4: రసీదుపై సంతకం చేసి ఏజెంట్ టూల్ కాల్‌ను చుట్టండి

ఒక వాస్తవ అమల్లో, మీరు ప్రతి ఏజెంట్ రచయిత `make_receipt`ని పిలవాలని గుర్తుంచుకోవాలనుకునరు. ప్రతి టూల్ వినంతి కోసం రసీదు సంతకం ఆటోమేటిక్ కావాలి.

ఇక్కడ అత్యంత సింపుల్ నమూనా ఉంది: ఏదైనా కాల్ చేయగల టూల్ ఫంక్షన్ తీసుకుని దాని నుండి రసీదు విడుదల చేసే వెర్షన్‌ను తిరిగి ఇచ్చే రాపర్ క్లాస్. ఇది Microsoft Agent Framework (`agent_framework.foundry`) సహా ఏ ఏజెంట్ ఫ్రేమ్‌వర్క్‌కు సరిపోతుంది.

మీరు Microsoft Foundry ప్రాజెక్ట్ సెట్ చేయకుంటే, దిగువ లోకల్ మాక్ ఈ నమూనాను ఇప్పటికీ ప్రదర్శిస్తుంది.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్‌తో సమీకరణ

పై `ReceiptedTool` రాపర్ ఫ్రేమ్‌వర్క్-నిరపేక్షంగా ఉంది. దీన్ని మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్‌తో నిర్మించబడిన ఏజెంట్‌లో ఉపయోగించాలంటే, రాప్ట్ ఫంక్షన్‌ను టూల్‌గా నమోదు చేయండి. ఒక స్కెచ్ (మీరు మాక్‌ను నిజమైన మైక్రోసాఫ్ట్ ఫౌండ్రీ టూల్ రిజిస్ట్రేషన్‌తో మార్చుతారు):

```python
# సమీకరణ ఆకారం చూపించే ప్సూడోకోడ్.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="మీరు Contoso Travel ఏజెంట్ ...",
#     tools=[receipted_lookup],   # అలంకరించిన టూల్, ముల ఫంక్షన్ కాదు
# )
# response = agent.run("జూన్‌లో సిడ్ని నుండి లాస్ ఏంజిల్స్ కి విమానాలు కనుగొనండి.")
#
# # రన్ తర్వాత, ఏజెంట్ చేసిన ప్రతి టూల్ కాల్ కు సంతకం చేసిన రసీదు ఉంటుంది:
# audit_chain = receipted_lookup.receipts
```

ఏజెంట్ ఫ్రేమ్‌వర్క్ రిసిప్ట్స్ గురించి ఏమీ తెలుసుకోవాల్సిన అవసరం లేదు. రిసిప్ట్ సంతకం టూల్ చుట్టూ రాప్డ్ చేయబడింది, ఫ్రేమ్‌వర్క్‌లో అమర్చబడలేదు. ఇది ఏజెంట్‌ను పునఃరాయించకుండా ప్రస్తుత ఏజెంట్ కోడ్‌కు ప్రొవెనెన్స్ ఎలా జోడించాలో చూపిస్తుంది.


## సారాంశం మరియు పొడిగింపు సవాలు

మీరు:

- Ed25519 కీ జతను సృష్టించారు.
- ఏజంట్ టూల్ కాల్‌కు రసీదు తయారు చేసి సంతకం చేశారు.
- పబ్లిక్ కీ మాత్రమే ఉపయోగించి రసీదును ఆఫ్‌లైన్‌లో నిర్ధారించుకున్నారు.
- ఒక రసీదును తల్లిపారేశారు మరియు ధృవీకరణ విఫలం అయినదని గమనించారు.
- మూడు రసీదుల హ్యాష్-చెయిన్ క్రమాన్ని నిర్మించారు.
- చెయిన్ మధ్యలో తల్లిపారేశారు మరియు సంతకం విఫలం మరియు చెయిన్-లింక్ విఫలత రెండింటినీ గమనించారు.
- ఆటోమాటిక్ రసీదు సంతకం తో ఒక టూల్ ఫంక్షన్‌ను ర్యాప్ చేశారు.

**పొడిగింపు సవాలు.** రసీదు స్కీమాను `request_id` ఫీల్డ్ (విభజిత ట్రేసింగ్ కోసం UUID)తో విస్తరించండి. దీన్ని చేర్చేందుకు `make_receipt` ను నవీకరించండి మరియు రసీదులు చివర వరకు నిర్ధారించబడతాయని ధృవీకరించండి. తరువాత సంతకం చేసిన తర్వాత ఆ ఫీల్డ్‌ను మార్చి ధృవీకరణ విఫలమవుతుందని నిర్ధారించండి. ఇది ప్రతి కేనానికల్ ఎంకోడింగ్ బైట్ సంతకానికి ఎలా సహాయపడుతుందో మీరు అంతర్గతంగా అర్థం చేసుకోవడాన్ని బలపరుస్తుంది.

** ముఖ్యమైన సరిహద్దు.** రసీదులు మూడు విషయాలను మాత్రమే నిరూపిస్తాయి: బాధ్యత (ఈ కీ ఈ విషయాన్ని సంతకం చేసింది), సమగ్రత (సంతకం చేసినప్పటి నుండి విషయం మారలేదు), మరియు క్రమం (ఈ రసీదు ఆ రసీదు తర్వాత వచ్చింది). అవి ఏజంట్ చర్య సరి అనేది, `policy_id` లోని పాలసీ నిజంగా మొత్తంగా అంచనా వేయబడిందా, లేదా ఏజంట్ ప్రతి నియమాన్ని పాటించిందా అని నిరూపించవు. రసీదులు ఒక పునాది. పాలన మీరు నిర్మించే వ్యవస్థ.

ఆ సరిహద్దు మనసులో పెట్టుకొని పాఠం READMEని మళ్లీ చదవండి. రసీదులతో టీమ్స్ చేస్తారు కనీసపు తప్పిదం "మాకు రసీదులు ఉన్నాయి అంటే మనం పాలనలో ఉన్నాము" అనుకోవడం. అలా కాదు. రసీదులు ఏజంట్ ప్రవర్తనను పరిశీలనీయంగా చేస్తాయి. అవి దానిని సరిగా మార్చవు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
